You do **not** need the whole repo. Batch 2 needs one folder with three children:

```
raag-identifier/                   <- point REPO at this
  utils/                           the shared package: config, dataset, raagdb,
                                   raagspace, musical_eval, extract
  survey-aug-2026/                 the whole folder (code only; cache/ and results/ are built here)
  hindustani-raag-small-v1.1/      built by cell 3; do not copy it, it is 620 MB
```

`motif-classifier/` is **not** needed on this path any more — what Batch 2 used from it now
lives in `utils/`. Only the Stage 4 DB prior still reaches into it, and that is not Batch 2.

Paths are configurable rather than assumed: `utils.config` reads `RAAG_DATASET_DIR`,
`RAAG_CACHE_DIR`, `RAAG_MELODY_DIR` and `RAAG_SEPARATION_DIR` if you set them, so the names
above are defaults, not requirements.

Then set `Runtime > Change runtime type > GPU`.

## 1. GPU and dependencies


In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU'


In [ ]:
%pip install -q 'transformers>=4.39,<5' 'datasets>=2.18' librosa soxr libmogra rapidfuzz
print('deps ok')  # huggingface_hub + pyarrow come with datasets


## 2. Mount Drive and locate the repo

`REPO` must be the folder that contains `raag-identifier/`. Adjust the path if you put it
somewhere else in Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# The one thing you set. It is the folder holding utils/ and survey-aug-2026/ -- put it
# wherever you like in Drive; this notebook can sit inside it or anywhere else.
REPO = Path('/content/drive/MyDrive/icm-shruti-analysis/raag-identifier')

SURVEY = REPO / 'survey-aug-2026'
missing = [str(d) for d in (REPO / 'utils', SURVEY / 'common', SURVEY / 'scripts')
           if not d.exists()]
if missing:
    raise FileNotFoundError('REPO is not the right folder -- missing:\n  ' +
                            '\n  '.join(missing))
sys.path[:0] = [str(REPO), str(SURVEY)]     # `import utils`, `from common import ...`

import utils                                 # fails HERE, not an hour in, if incomplete
print('utils :', Path(utils.__file__).parent)
print('survey:', SURVEY)

## 3. Materialise the dataset at the pinned revision

v1.1 audio lives **only in the parquet files** on the Hub; the `<Raag>/*.mp3` tree at the
dataset root is still v0. `fetch_dataset.py` reads the parquet and writes the layout the
rest of the pipeline expects, pinned to the exact commit so a later push to the dataset
cannot silently change what a result was computed on.

~620 MB. Written to Drive, so this is a one-time cost across all your Colab sessions.


In [ ]:
from utils import dataset

DATA = REPO / 'hindustani-raag-small-v1.1'
REVISION = '326caef0bc01da44ad46e4d9c65a5146da6bcc5b'

if (DATA / 'REVISION').exists():
    print('already present:', (DATA / 'REVISION').read_text().strip())
else:
    dataset.fetch(
        repo_id='neerajaabhyankar/hindustani-raag-small',
        revision=REVISION,
        audio_dir=DATA,
        tonics_csv=DATA / 'tonics.csv',
        revision_file=DATA / 'REVISION',
    )

## 4. Build the audio cache

Decode 1960 clips to 22.05 kHz int16 plus the two CQT variants, ~2.3 GB, onto Drive.
**This is the cell that has crashed before.** It is resumable: if it dies, just run it
again and it continues from where it stopped. Drive I/O makes it slower than on a laptop
(expect 15-25 min the first time, seconds every time after).


In [ ]:
!cd {SURVEY} && PYTHONPATH={REPO} python scripts/00_build_cache.py --workers 2


## 4b. Preflight the graded metrics

`musical_eval` needs the class list. It now reads it from the dataset's own `tonics.csv`
via `utils.dataset.raag_names`, so — unlike before — there is no dependency on a *second*,
older copy of the dataset being on disk for its directory names. This cell just proves the
chain imports before you spend an hour of GPU on a run that ends in metrics.

In [ ]:
# nothing to stub any more -- utils.raagdb reads the class list from the dataset's
# own tonics.csv. Kept as a check that the graded-metrics chain imports.
from utils import dataset as _ds
from utils.raagspace import affinity

names = _ds.raag_names(tonics_csv=DATA / 'tonics.csv')
labels, A, A_rot, best_k = affinity(names=names)
print(f'{len(names)} classes, affinity matrix {A.shape} -- libmogra/raagspace/raagdb all resolve')

## 5. Run Batch 2

The three Stage 1-2 distilHuBERT runs: the original recipe on v1.1, then the tonic
normalised into the audio, then the tonic supplied by FiLM. Batch size is raised from 8
to 16 because a GPU has the memory for it; everything else matches the laptop script.

Watch the per-epoch lines. If the runtime disconnects, re-run this cell.


In [ ]:
!cd {SURVEY} && PYTHONPATH={REPO} python scripts/10_train.py --arch hubert --run-id d1  --stage 1 \
    --epochs 20 --patience 6 --batch-size 16 --select-on top1 --device cuda



In [ ]:
!cd {SURVEY} && PYTHONPATH={REPO} python scripts/10_train.py --arch hubert --run-id d2n --stage 2 \
    --tonic normalise --epochs 20 --patience 6 --batch-size 16 --select-on top1 --device cuda



In [ ]:
!cd {SURVEY} && PYTHONPATH={REPO} python scripts/10_train.py --arch hubert --run-id d2c --stage 2 \
    --tonic-mode condition --epochs 20 --patience 6 --batch-size 16 --select-on top1 --device cuda



## 6. Optional — unfreeze the convolutional feature encoder

On the M1 this is the difference between 12 minutes and 5.5 hours per epoch, so it is off
by default. On a GPU it is affordable, and it is the one recipe knob the laptop cannot
explore. Worth one run to find out whether the frozen front end was costing anything.


In [ ]:
!cd {SURVEY} && PYTHONPATH={REPO} python scripts/10_train.py --arch hubert --run-id d1_unfrozen --stage 1 \
    --unfreeze-encoder --epochs 20 --patience 6 --batch-size 8 --lr 3e-5 --select-on top1 --device cuda



## 7. Report

Regenerates `results/v1.1/RESULTS.md` from every `result.json` on Drive — including the
runs done on the laptop, since both write to the same folder. Paste the table back into
the chat, or just say which run ids finished.


In [ ]:
!cd {SURVEY} && PYTHONPATH={REPO} python scripts/90_report.py --write
